# Mult-VAE Autoencoder — UC1
Runner notebook (logic lives in `book_recsys.models.autoencoder`). Trained on the **same 30k users / 2,273,496 interactions as SASRec**.
Edit the **config cell** and re-run to try a different setting.

In [ ]:
# --- config cell: edit these, then Run All ---
N_USERS, MAX_HIST, EXPECT_ROWS = 30000, 100, 2_273_496
LATENT, HIDDEN, BETA_CAP, DROPOUT = 200, 600, 0.2, 0.5
EPOCHS, LR, MIN_ITEM_COUNT, POP_DISCOUNT = 150, 1e-3, 1, 0.0
ANNEAL_STEPS = 3000   # beta reaches BETA_CAP by ~epoch 50 and holds (60 batches/epoch)
DEVICE, CKPT_DIR, N_EVAL_USERS = None, 'artifacts', 2000

In [ ]:
import pandas as pd
from book_recsys.data.splits import leave_last_n_out
from book_recsys.data.schema import BOOK
from book_recsys.models.autoencoder.data import reproduce_sasrec_sample
from book_recsys.models.autoencoder.recommender import MultVaeRecommender
from book_recsys.eval.harness import (build_user_histories, build_relevance,
    evaluate_sampled_negatives, popularity_diagnostics)
sample = reproduce_sasrec_sample(pd.read_parquet('artifacts/sample.parquet'),
    N_USERS, MAX_HIST, expect_rows=EXPECT_ROWS)
train, test = leave_last_n_out(sample, n=1)
print(sample['user_id'].nunique(), 'users,', len(sample), 'interactions')

In [ ]:
rec = MultVaeRecommender(hidden=HIDDEN, latent=LATENT, dropout=DROPOUT,
    beta_cap=BETA_CAP, epochs=EPOCHS, lr=LR, min_item_count=MIN_ITEM_COUNT,
    pop_discount=POP_DISCOUNT, device=DEVICE, ckpt_dir=CKPT_DIR,
    anneal_steps=ANNEAL_STEPS, progress=True).fit(train)

In [ ]:
histories = build_user_histories(train)
relevance = build_relevance(test)
eval_users = list(relevance)[:N_EVAL_USERS]
relevance = {u: relevance[u] for u in eval_users}
all_items = sample[BOOK].unique()
weights = sample[BOOK].value_counts().reindex(all_items).to_numpy()
headline = evaluate_sampled_negatives(rec, histories, relevance, all_items,
    n_neg=100, k=10, seed=0, item_weights=weights)
headline

In [ ]:
# Anti-popularity diagnostics + the FREE alpha serendipity sweep (no retraining)
pop_order = list(sample[BOOK].value_counts().index)
diag = popularity_diagnostics(rec, {u: histories.get(u, []) for u in eval_users},
    pop_order, catalog_size=len(all_items), k=10)
rows = []
for a in (0.0, 0.25, 0.5, 1.0):
    rec.pop_discount = a
    m = evaluate_sampled_negatives(rec, histories, relevance, all_items,
        n_neg=100, k=10, seed=0, item_weights=weights)
    rows.append({'alpha': a, **m})
rec.pop_discount = POP_DISCOUNT
import pandas as pd
print(diag); pd.DataFrame(rows)